# LLM


## Import Data

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Load the CSV file into a DataFrame with the correct encoding
decentraland= pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/Decentraland_Community_general.csv', encoding='ISO-8859-1')

# Assume decentraland is the DataFrame you want to remove unnamed columns from
decentraland = decentraland.loc[:, ~decentraland.columns.str.startswith('Unnamed:')]

In [ ]:
# Date format change
decentraland['Date'] = pd.to_datetime(decentraland['Date'], format='%Y-%m-%dT%H:%M:%S.%f%z')
decentraland['Date'] = decentraland['Date'].dt.strftime('%m/%d/%Y %I:%M %p')

In [ ]:
decentraland.head()

,AuthorID,Author,Date,Content,Attachments,Reactions
0,2.890000e+17,jtv.,06/15/2023 01:44 AM,"Hey, does anyone know where I can find the Dec...",NaN,NaN
1,3.190000e+17,toonpunk,06/15/2023 02:01 AM,"this one - https://genesis.city/#0,0,3z",NaN,"dcl (2),dclgm (1)"
2,2.890000e+17,jtv.,06/15/2023 02:06 AM,Yes thank you!,NaN,NaN
3,1.020000e+18,Decentraland#7255,06/15/2023 08:00 AM,NaN,NaN,NaN
4,1.080000e+18,haroonsaqib.,06/15/2023 07:14 PM,"Hello eveyone, I just wanted to know which mul...",NaN,NaN


In [ ]:
decentraland.tail()

,AuthorID,Author,Date,Content,Attachments,Reactions
5878,7.140000e+17,peterzheng99,06/13/2024 11:15 PM,Hi! How do we buy land in Decentraland?,NaN,NaN
5879,3.500000e+17,.vitsky,06/13/2024 11:17 PM,You can buy lands on our Marketplace ð http...,NaN,NaN
5880,7.810000e+17,eldanak_dcl,06/14/2024 01:24 AM,# ð¥ **Feature Focus Thursday** ð¥\n\nWe w...,NaN,ð¥ (3)
5881,1.020000e+18,Decentraland#7255,06/14/2024 08:00 AM,NaN,NaN,NaN
5882,3.990000e+17,cybermike.,06/14/2024 04:51 PM,@Kaze_no_Kai scam,NaN,"â¤ï¸ (3),ð (3)"


## Text Processing

In [ ]:
# Delete blank rows in the "Content" column.
decentraland_cleaned = decentraland.dropna(subset=['Content'])

# Save the cleaned DataFrame to a new CSV file
decentraland_cleaned.to_csv('message_cleaned.csv', index=False)


# Sentiment Analysis using LLM

## Sentiment Label and Sentiment Score

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Read CSV file from GitHub
url = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/D/message_cleaned.csv'
df = pd.read_csv(url)

# Extract data from "Content" column
content_data = df['Content'].tolist()

# Load the tokenizer
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Maximum sequence length for the model
max_length = 512

# Filter out texts longer than max_length
filtered_content_data = []
filtered_indices = []

for i, text in enumerate(content_data):
    tokens = tokenizer.encode(text, max_length=max_length, truncation=True)
    if len(tokens) <= max_length:
        filtered_content_data.append(text)
        filtered_indices.append(i)
    print(f"Text length: {len(tokens)}, included: {len(tokens) <= max_length}")

# Create a new DataFrame with the filtered data
filtered_df = df.iloc[filtered_indices].copy()

# Save the filtered data to a new CSV file
filtered_df.to_csv('filtered_message.csv', index=False)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Streaming output truncated to the last 5000 lines.
Text length: 35, included: True
Text length: 73, included: True
Text length: 3, included: True
Text length: 28, included: True
Text length: 8, included: True
Text length: 4, included: True
Text length: 28, included: True
Text length: 22, included: True
Text length: 16, included: True
Text length: 4, included: True
Text length: 3, included: True
Text length: 10, included: True
Text length: 7, included: True
Text length: 36, included: True
Text length: 13, included: True
Text length: 3, included: True
Text length: 40, included: True
Text length: 38, included: True
Text length: 20, included: True
Text length: 39, included: True
Text length: 21, included: True
Text length: 5, included: True
Text length: 21, included: True
Text length: 26, included: True
Text length: 17, included: True
Text length: 24, included: True
Text length: 42, included: True
Text length: 19, included: True
Text length: 12, included: True
Text length: 26, included: Tr

In [ ]:
import pandas as pd
from transformers import pipeline

# Read the filtered CSV file
filtered_df = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/filtered_message_1.csv')

# Extract data from "Content" column
filtered_content_data = filtered_df['Content'].tolist()

# Define the sentiment analysis model to use
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

try:
    # Load the sentiment analysis pipeline
    sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

    # Perform sentiment analysis
    results = sentiment_pipeline(filtered_content_data)

    # Extract labels and scores
    labels = [result['label'] for result in results]
    scores = [result['score'] for result in results]

    # Add results to the dataframe
    filtered_df['Label'] = labels
    filtered_df['Score'] = scores

    # Save results to a new CSV file
    filtered_df.to_csv('message_with_sentiment.csv', index=False)

    # Output to verify
    print(filtered_df.head())
except Exception as e:
    print(f"An error occurred with model {model_name}: {e}")


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


An error occurred with model cardiffnlp/twitter-roberta-base-sentiment-latest: The expanded size of the tensor (861) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 861].  Tensor sizes: [1, 514]


In [ ]:
import pandas as pd
from transformers import pipeline

# Read the filtered CSV file
filtered_df = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/filtered_message_1.csv')

# Extract data from the Content column
filtered_content_data = filtered_df['Content'].tolist()

# Define the sentiment analysis model to use
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

try:
    # Load the sentiment analysis pipeline
    sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

    # Perform sentiment analysis
    results = sentiment_pipeline(filtered_content_data)

    # Extract labels and scores
    labels = [result['label'] for result in results]
    scores = [result['score'] for result in results]

    # Add the results to the DataFrame
    filtered_df['Label'] = labels
    filtered_df['Score'] = scores

    # Save the results to a new CSV file
    filtered_df.to_csv('message_with_sentiment.csv', index=False)

    # Output to verify
    print(filtered_df.head())
except Exception as e:
    print(f"An error occurred with model {model_name}: {e}")
    print(f"Error details: {str(e)}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

An error occurred with model cardiffnlp/twitter-roberta-base-sentiment-latest: The expanded size of the tensor (861) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 861].  Tensor sizes: [1, 514]
Error details: The expanded size of the tensor (861) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 861].  Tensor sizes: [1, 514]


In [ ]:
import pandas as pd
from transformers import pipeline

# Read the filtered CSV file
filtered_df = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/filtered_message_1.csv')

# Extract data from the Content column
filtered_content_data = filtered_df['Content'].tolist()

# Define the sentiment analysis model to use
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Load the sentiment analysis pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

# Create lists to store labels and scores
labels = []
scores = []

for i, content in enumerate(filtered_content_data):
    try:
        # Perform sentiment analysis
        result = sentiment_pipeline(content)[0]

        # Extract label and score
        labels.append(result['label'])
        scores.append(result['score'])
    except Exception as e:
        print(f"An error occurred with model {model_name} on message index {i}: {e}")
        print(f"Message content: {content}")
        # Add None values when an error occurs
        labels.append(None)
        scores.append(None)

# Add the results to the DataFrame
filtered_df['Label'] = labels
filtered_df['Score'] = scores

# Save the results to a new CSV file
filtered_df.to_csv('message_with_sentiment.csv', index=False)

# Output to verify
print(filtered_df.head())

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


An error occurred with model cardiffnlp/twitter-roberta-base-sentiment-latest on message index 3025: The expanded size of the tensor (861) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 861].  Tensor sizes: [1, 514]
Message content: ð PrÃ©-Save ImperdÃ­vel: "De Frente para o Abismo" - Clipe de Salem Mc

Salve, galera animada! ð

EstÃ¡ chegando o grande momento que todos esperavam: o lanÃ§amento do clipe "De Frente para o Abismo" de Salem Mc! ð¥ VocÃª nÃ£o vai querer perder esse evento incrÃ­vel que estÃ¡ marcado para amanhÃ£, Ã s 23h, no YouTube.

ð¶ Letra Que Promete Mexer com a CabeÃ§a:

"Sai da caverna, PlatÃ£o me admira
E eu tÃ´ aqui pra te mostrar a verdadeira ira,
De um cara revoltado com essa cena do pÃ³
Na cidade, corrompendo altas minas e os menor..."

SÃ£o versos que vÃ£o te fazer refletir, viajar nas palavras e se conectar com a mensagem forte dessa mÃºsica. E acredite, o clipe estÃ¡ Ã  altura!

Mas aqui vai um toque: nÃ£o Ã© sÃ

In [ ]:
import pandas as pd
from transformers import pipeline

# Read the filtered CSV file
filtered_df = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/filtered_message.csv')

# Extract data from the Content column
filtered_content_data = filtered_df['Content'].tolist()

# Define the sentiment analysis model to use
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Load the sentiment analysis pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

# Create lists to store labels and scores
labels = []
scores = []

for i, content in enumerate(filtered_content_data):
    try:
        # Perform sentiment analysis
        result = sentiment_pipeline(content)[0]

        # Extract label and score
        labels.append(result['label'])
        scores.append(result['score'])
    except Exception as e:
        print(f"An error occurred with model {model_name} on message index {i}: {e}")
        print(f"Message content: {content}")
        # Add None values when an error occurs
        labels.append(None)
        scores.append(None)

# Add the results to the DataFrame
filtered_df['Label'] = labels
filtered_df['Score'] = scores

# Save the results to a new CSV file
filtered_df.to_csv('message_with_sentiment.csv', index=False)

# Output to verify
print(filtered_df.head())

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


       AuthorID        Author                 Date  \
0  2.890000e+17          jtv.  06/15/2023 01:44 AM   
1  3.190000e+17      toonpunk  06/15/2023 02:01 AM   
2  2.890000e+17          jtv.  06/15/2023 02:06 AM   
3  1.080000e+18  haroonsaqib.  06/15/2023 07:14 PM   
4  1.080000e+18  haroonsaqib.  06/15/2023 07:15 PM   

                                             Content Attachments  \
0  Hey, does anyone know where I can find the Dec...         NaN   
1            this one - https://genesis.city/#0,0,3z         NaN   
2                                     Yes thank you!         NaN   
3  Hello eveyone, I just wanted to know which mul...         NaN   
4   I am developer and want to deploy a clients land         NaN   

           Reactions     Label     Score  
0                NaN   neutral  0.926633  
1  dcl (2),dclgm (1)   neutral  0.855839  
2                NaN  positive  0.968376  
3                NaN   neutral  0.773710  
4                NaN   neutral  0.613514  


## Daily Average Sentiment

In [ ]:
import pandas as pd

# Read data from file
file_path = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/D/message_with_sentiment_1.csv'
df = pd.read_csv(file_path)

# Check column names
print(df.columns)

# Define function to calculate sentiment score
def calculate_sentiment_score(label, score):
    if label == 'positive':
        return score
    elif label == 'negative':
        return -score
    else:
        return 0

# Calculate sentiment scores
df['Sentiment_Score'] = df.apply(lambda row: calculate_sentiment_score(row['Label'], row['Score']), axis=1)

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date']).dt.date

# Calculate daily average sentiment scores
daily_sentiment = df.groupby('Date')['Sentiment_Score'].mean().reset_index()

# Define labeling
def label_sentiment(score):
    if score > 0.1:  # Define threshold for positive sentiment
        return 'positive'
    elif score < -0.1:  # Define threshold for negative sentiment
        return 'negative'
    else:
        return 'neutral'

# Assign labels to daily average sentiment scores
daily_sentiment['Label'] = daily_sentiment['Sentiment_Score'].apply(label_sentiment)

# Save results to a new CSV file
output_file_path = 'daily_sentiment_scores.csv'  # Replace with the path where you want to save it
daily_sentiment.to_csv(output_file_path, index=False)

# Output to verify
print(daily_sentiment.head())


Index(['AuthorID', 'Author', 'Date', 'Content', 'Attachments', 'Reactions',
       'Label', 'Score'],
      dtype='object')
         Date  Sentiment_Score     Label
0  2023-06-15         0.138339  positive
1  2023-06-16         0.067203   neutral
2  2023-06-17         0.097982   neutral
3  2023-06-18         0.111416  positive
4  2023-06-19         0.195899  positive


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Define the path to the CSV file
file_path = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/D/message_with_sentiment_1.csv'

# Read the CSV file
data = pd.read_csv(file_path)

# Ensure the date column is in datetime format
data['Date'] = pd.to_datetime(data['Date'], format='%m/%d/%Y %I:%M %p')

# Set the date range
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 6, 14)
date_range = pd.date_range(start_date, end_date)

# Check for missing dates
existing_dates = data['Date'].dt.date.unique()
missing_dates = set(date_range.date) - set(existing_dates)

print(f"Missing dates: {sorted(missing_dates)}")

Missing dates: [datetime.date(2024, 4, 28), datetime.date(2024, 5, 13)]


In [ ]:
import pandas as pd
from datetime import datetime

# Define the path to the CSV file
file_path = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/D/daily_sentiment_scores.csv'

# Read the CSV file
data = pd.read_csv(file_path)

# Ensure the date column is in datetime format
data['Date'] = pd.to_datetime(data['Date'], format='%Y-%m-%d')

# Fill in missing dates with data
missing_dates = [datetime(2024, 4, 28), datetime(2024, 5, 13)]
new_rows = [{'Date': date, 'Sentiment_Score': 0.0, 'Label': 'neutral'} for date in missing_dates]

# Use pd.concat to add new rows
data = pd.concat([data, pd.DataFrame(new_rows)], ignore_index=True)

# Keep the score rounded to 4 decimal places
data['Sentiment_Score'] = data['Sentiment_Score'].round(4)

# Sort by date
data = data.sort_values(by='Date')

# Save to a new CSV file
new_file_path = 'daily_processed_scores.csv'
data.to_csv(new_file_path, index=False)

# Figure

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Read CSV file from GitHub
file_path = "https://raw.githubusercontent.com/Xintong1122/KDD/main/D/message_with_sentiment_1.csv"
data = pd.read_csv(file_path)

# Extract relevant columns
labels = ['positive', 'neutral', 'negative']

# Calculate the number of each sentiment label for LLM1
llm1_counts = [(data['Label'] == label).sum() for label in labels]

# Prepare text to display on each bar and set font settings
llm1_text = [f'{count}' for count in llm1_counts]

# Custom colors for sentiments in hexadecimal format
colors = {'positive': '#66c2a5', 'neutral': '#8da0cb', 'negative': '#fb8d62'}

# Plot bar chart for LLM1
fig = go.Figure(data=[
    go.Bar(name='LLM', x=labels, y=llm1_counts, text=llm1_text, textposition='auto', textfont_color='white', marker_color=[colors[label] for label in labels])
])

# Modify layout and style
fig.update_layout(
    font=dict(family="Times New Roman, Times, serif", size=14),
    xaxis_title='Sentiment Label',
    yaxis_title='Number',
    height=625,
    width=1000,
    title='Distribution of Sentiment Categories using LLM',
)

# Show the chart
fig.show()


In [ ]:
import pandas as pd
import plotly.express as px

# Read CSV file from GitHub
url = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/D/2.csv'
df = pd.read_csv(url)

# Rename columns
df = df.rename(columns={
    'Sentiment_Score': 'LLM1',
})

# Plot line chart for LLM1
fig = px.line(df, x='Date', y='LLM1',
              labels={'value': 'Sentiment Score', 'variable': 'LLM Model'},
              title='Sentiment Scores over Time')

# Set line color to orange and style
fig.update_traces(line=dict(color='orange', width=2), selector=dict(mode='lines'))

# Update font style and axis annotations
fig.update_layout(
    font=dict(family="Times New Roman, Times, serif", size=12),
    xaxis=dict(title='Date(d)', tickformat='%b%d', tickfont=dict(size=12)),
    yaxis=dict(title='Sentiment Score', tickfont=dict(size=12)),
    legend=dict(title='LLM Model', title_font=dict(size=12)),
    height=625,
    width=1000,
    margin=dict(l=50, r=50, t=50, b=50),
)

# Show the chart
fig.show()
